# Estudo de Caso: atendimentos em UBS

Nesta parte da aula vamos juntar tudo o que aprendemos (importar, limpar, filtrar, criar variáveis, fazer gráficos e exportar) em um único exemplo, do começo ao fim.

**A base:** 40 registros de atendimentos em três UBS (Central, Norte e Sul). Ela é **fictícia** e foi feita "suja" de propósito: tem linhas duplicadas, valores ausentes, valores impossíveis (outliers) e texto mal formatado. É o tipo de problema que aparece o tempo todo em dados reais.

**Como usar este notebook**
- Rode as células em ordem, uma de cada vez (`Shift + Enter`).
- Antes de rodar uma célula de código, tente adivinhar o que vai aparecer.
- Os quadros **Para pensar** não têm uma única resposta certa. São decisões que quem analisa os dados precisa tomar.
- No final tem uma seção **Sua vez** com alguns exercícios.

**Roteiro**
1. Importação e inspeção
2. Duplicatas
3. Dados ausentes
4. Outliers
5. Formatação, filtros e novas variáveis
6. Gráficos
7. Verificação final e exportação

## Dicionário de dados

Antes de mexer em qualquer base, a gente precisa saber o que cada coluna significa. Quem faz isso é o **dicionário de dados**.

| Variável | Tipo | Descrição |
|---|---|---|
| NOME | Texto | Nome do paciente |
| SEXO | Categórica | Sexo (F/M; **9 = ignorado**) |
| IDADE | Numérica | Idade em anos |
| PESO_KG | Numérica | Peso em quilogramas |
| UBS | Categórica | Unidade Básica de Saúde do atendimento |
| DT_ATENDIMENTO | Data | Data do atendimento |

Guardem o detalhe do **9 = ignorado** na coluna SEXO. Ele vai voltar mais pra frente.

## Antes de começar: levando o arquivo para o Colab

O Colab roda em um computador na nuvem, então ele não enxerga os arquivos que estão no seu computador. Precisamos enviar o `pacientes.csv` para lá. Tem duas formas:

1. Pelo ícone de **pasta** na barra lateral esquerda, depois no botão de upload.
2. Rodando a célula abaixo, que abre um botão **Escolher arquivos**.

Atenção: o arquivo precisa se chamar exatamente `pacientes.csv`. Se o navegador renomear para algo como `pacientes (1).csv`, renomeie antes de continuar.

In [ ]:
# Roda só no Google Colab: abre o botão para enviar o arquivo
from google.colab import files
enviado = files.upload()

**Plano B (se o upload não funcionar):** a célula abaixo cria o `pacientes.csv` direto no Colab, com o mesmo conteúdo. Só use se der problema no upload.

In [ ]:
%%writefile pacientes.csv
NOME;SEXO;IDADE;PESO_KG;UBS;DT_ATENDIMENTO
Maria da Silva;F;34.0;68.5;UBS Central;2026-03-02
Joao Pedro Souza;M;45.0;82.0;UBS Central;2026-03-02
Ana Paula Lima;F;200.0;75.3;UBS Norte;2026-03-03
Carlos Eduardo;M;29.0;; UBS Sul;2026-03-03
  Francisca Alves  ;F;61.0;59.8;UBS Norte;2026-03-04
Jose Ferreira;M;;90.1;UBS Central;2026-03-04
Maria da Silva;F;34.0;68.5;UBS Central;2026-03-02
Rita de Cassia;F;52.0;71.2;UBS Sul;2026-03-05
Antonio Carlos;M;8.0;30.4;UBS Norte;2026-03-05
Juliana Santos;F;27.0;62.0;UBS Central;2026-03-06
Pedro Henrique;M;39.0;77.5;UBS Sul;2026-03-06
  Marcos Vinicius;M;71.0;0.0;UBS Norte;2026-03-07
Sandra Regina;F;44.0;65.9;UBS Central;2026-03-07
Luiz Gonzaga;M;;88.3;UBS Sul;2026-03-08
Patricia Gomes;F;19.0;54.2;UBS Norte;2026-03-08
Roberto Carlos;M;33.0;79.0;UBS Central;2026-03-09
Fernanda Dias;F;58.0;61.7;UBS Sul;2026-03-09
Joao Pedro Souza;M;45.0;82.0;UBS Central;2026-03-02
Camila Ribeiro;F;22.0;58.4;UBS Norte;2026-03-10
Bruno Costa;M;47.0;95.6;UBS Central;2026-03-10
Vera Lucia;F;66.0;66.3;UBS Sul;2026-03-11
Marcos Vinicius;M;71.0;84.1;UBS Norte;2026-03-07
Aline Cristina;F;31.0;63.5;UBS Central;2026-03-12
Rafael Nunes;M;15.0;45.0;UBS Sul;2026-03-12
Silvia Helena;F;49.0;70.2;UBS Norte;2026-03-13
Eduardo Pires;M;-5.0;78.9;UBS Central;2026-03-13
Tatiane Moraes;F;60.0;60.1;UBS Sul;2026-03-14
Gabriel Rocha;M;26.0;33.7;UBS Norte;2026-03-14
Larissa Melo;F;38.0;57.8;UBS Central;2026-03-15
Diego Fernandes;M;42.0;81.4;UBS Sul;2026-03-15
Monica Barbosa;F;55.0;69.9;UBS Norte;2026-03-16
Thiago Araujo;M;30.0;74.6;UBS Central;2026-03-16
Cristina Farias;F;24.0;;UBS Sul;2026-03-17
Vinicius Teixeira;M;68.0;87.2;UBS Norte;2026-03-17
Adriana Castro;F;37.0;64.0;UBS Central;2026-03-18
Renato Lopes;M;41.0;76.3;UBS Sul;2026-03-18
Beatriz Cardoso;F;12.0;40.2;UBS Norte;2026-03-19
Felipe Andrade;M;53.0;80.5;UBS Central;2026-03-19
Simone Ramos;F;46.0;67.8;UBS Sul;2026-03-20
Wagner Duarte;9;63.0;72.1;UBS Norte;2026-03-20

## 0. Bibliotecas

Vamos chamar três bibliotecas:

- **pandas** (`pd`): trabalha com tabelas de dados (os DataFrames);
- **numpy** (`np`): vamos usar só para representar valores ausentes, com `np.nan`;
- **matplotlib** (`plt`): faz os gráficos.

Os apelidos (`pd`, `np`, `plt`) são só para não precisar digitar o nome inteiro toda vez.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## 1. Importação e inspeção

### 1.1 Lendo o arquivo

Usamos `pd.read_csv()`. O arquivo é CSV, mas as colunas estão separadas por **ponto e vírgula** (`;`) e não por vírgula. Isso é muito comum em bases brasileiras, porque aqui a vírgula já é usada como separador decimal. Por isso precisamos avisar o pandas com `sep=';'`.

Guardamos a tabela em uma variável chamada `dados`.

In [ ]:
dados = pd.read_csv('pacientes.csv', sep=';')

### 1.2 Tamanho e colunas

`.shape` devolve **(número de linhas, número de colunas)**. `.columns` lista os nomes das colunas.

In [ ]:
print(dados.shape)
dados.columns

### 1.3 Informações gerais com `.info()`

Esse é o "raio-X" da tabela. Olhem com atenção para duas colunas do resultado:

- **Non-Null Count**: quantos valores *não vazios* existem em cada coluna. A tabela tem 40 linhas, então qualquer coluna com menos de 40 já indica dado ausente. Quais colunas são essas?
- **Dtype**: o tipo de cada coluna. `float64` é número com casas decimais; `object` (ou `str`, dependendo da versão) é texto.

Reparem que `IDADE` é `float64`, mesmo sendo uma idade em anos. Isso acontece porque a coluna tem valores vazios, e o pandas representa vazio como `NaN`, que é um número decimal.

In [ ]:
dados.info()

### 1.4 Espiando as linhas

- `.head()` mostra as 5 primeiras linhas;
- `.sample(5)` mostra 5 linhas aleatórias. Rode mais de uma vez e veja que muda. É bom para não olhar só o começo da tabela.

In [ ]:
dados.head()

In [ ]:
dados.sample(5)

## 2. Duplicatas

Pacientes cadastrados duas vezes inflam a contagem de atendimentos e distorcem qualquer análise. Vamos procurar linhas repetidas.

`.duplicated()` marca como `True` as linhas que já apareceram antes. Com `keep=False`, ele marca **todas** as linhas envolvidas na repetição (a original e as cópias), o que ajuda a enxergar o problema.

In [ ]:
dados[dados.duplicated(keep=False)]

Temos dois pacientes repetidos: **Maria da Silva** e **João Pedro Souza** (cada um aparece duas vezes, com todos os campos iguais).

Para remover, usamos `.drop_duplicates()`. Com `keep='first'` ficamos com a primeira ocorrência e descartamos as cópias.

In [ ]:
dados = dados.drop_duplicates(keep='first')
dados.shape

Conferindo: a base tinha **40** linhas e agora tem **38**. Foram removidas 2 duplicatas.

> **Para pensar:** o `.drop_duplicates()` sem nenhum argumento só remove linhas que são **idênticas em todas as colunas**. E se o mesmo paciente aparecer duas vezes com um campo ligeiramente diferente? Vamos encontrar um caso assim mais pra frente.

## 3. Dados ausentes

Na saúde é normal faltar preencher algum campo. O pandas marca esses buracos como `NaN` ("not a number"), e eles atrapalham contas e modelos.

### 3.1 Onde estão os ausentes?

`.isnull().sum()` conta quantos valores ausentes existem em cada coluna.

In [ ]:
dados.isnull().sum()

`IDADE` tem 2 ausentes e `PESO_KG` tem 2. E `SEXO` aparece com **0**. Será que está tudo certo com ela?

Lembram do dicionário de dados? O código **9** significa "ignorado", ou seja, a informação não foi coletada. Vamos olhar os valores diferentes que existem na coluna com `.unique()`.

In [ ]:
dados['SEXO'].unique()

Aí está: além de `F` e `M`, existe um `'9'`. Ele é um **código para "não sabemos"**, mas o pandas não tem como adivinhar isso, então conta como um valor válido. Se deixarmos assim, o 9 vira uma "categoria" de sexo nos gráficos e nas contagens.

A solução é **recodificar**: trocar o 9 por `NaN`.

Um detalhe importante: o 9 está entre aspas (`'9'`) porque a coluna inteira é texto (tem `F` e `M` misturados). Por isso o `.replace()` precisa receber `'9'` (texto) e não `9` (número). Se a gente escrevesse `replace(9, np.nan)`, nada seria trocado e ninguém avisaria que deu errado.

In [ ]:
dados['SEXO'] = dados['SEXO'].replace('9', np.nan)
dados.isnull().sum()

Agora o `SEXO` mostra 1 ausente. É o único paciente que tinha o código 9.

### 3.2 O que fazer com cada ausente?

Aqui não existe regra única. Quem decide é o analista, olhando para o tipo de variável:

- **SEXO**: é uma categoria, então não faz sentido "preencher". Deixamos como `NaN`.
- **PESO_KG**: é um número e podemos estimar um valor razoável. Vamos preencher com a **mediana**, que é menos sensível a valores extremos do que a média.
- **IDADE**: não temos como estimar a idade de uma pessoa com confiança, então vamos **remover** essas linhas.

In [ ]:
# PESO_KG ausente -> preenchido com a mediana
mediana_peso = dados['PESO_KG'].median()
print('Mediana do peso:', mediana_peso)
dados['PESO_KG'] = dados['PESO_KG'].fillna(mediana_peso)

In [ ]:
# IDADE ausente -> linhas removidas
dados = dados.dropna(subset=['IDADE'])
print(dados.shape)
dados.isnull().sum()

Agora só resta o `NaN` do `SEXO`, que deixamos de propósito. A base passou de 38 para **36** linhas (as 2 sem idade foram removidas).

Note o `subset=['IDADE']`: ele diz ao `.dropna()` para olhar **só** a coluna IDADE. Sem isso, ele removeria qualquer linha com qualquer campo vazio, inclusive a do SEXO.

## 4. Outliers

Outliers são valores que fogem muito do padrão. Podem ser **erros** (digitação, medição) ou **casos extremos reais**. Descobrir qual dos dois é o trabalho do analista.

Uma forma prática de encontrar: usar `.describe()` e comparar o **mínimo** e o **máximo** com a **mediana** (linha `50%`) e a média. Se estiverem muito distantes, é sinal de alerta.

In [ ]:
dados.describe()

Olhando a tabela:

- `IDADE`: o mínimo é **-5** e o máximo é **200**. Nenhum dos dois é possível. Repare também que a média (44,75) está acima da mediana (41,5), puxada pelo 200.
- `PESO_KG`: o mínimo é **0**. Ninguém pesa 0 kg, então é erro de registro.

A estratégia que vamos usar: transformar os valores impossíveis em `NaN` e depois preencher com a mediana. Para idade, vamos considerar impossível qualquer valor negativo ou acima de 110 anos.

In [ ]:
# IDADE negativa ou muito acima do plausível -> ausente
dados.loc[(dados.IDADE < 0) | (dados.IDADE > 110), 'IDADE'] = np.nan

# PESO_KG igual a 0 -> ausente (erro de registro)
dados.loc[dados.PESO_KG == 0, 'PESO_KG'] = np.nan

dados[['IDADE', 'PESO_KG']].isnull().sum()

Criamos 2 ausentes em `IDADE` (o -5 e o 200) e 1 em `PESO_KG` (o 0). Agora preenchemos com a mediana de cada coluna e olhamos o `.describe()` de novo.

In [ ]:
dados['IDADE'] = dados['IDADE'].fillna(dados['IDADE'].median())
dados['PESO_KG'] = dados['PESO_KG'].fillna(dados['PESO_KG'].median())
dados.describe()

Comparando **antes x depois**:

| | IDADE antes | IDADE depois | PESO_KG antes | PESO_KG depois |
|---|---|---|---|---|
| mínimo | -5 | 8 | 0 | 30,4 |
| máximo | 200 | 71 | 95,6 | 95,6 |
| média | 44,75 | 41,64 | 65,38 | 67,30 |

Os valores agora fazem sentido.

> **Para pensar:** o limite de 110 anos foi uma escolha nossa. Se fosse 90, o que mudaria? E o paciente de 26 anos com 33,7 kg? Passou pelas nossas regras, mas ainda assim parece estranho. Olhar o peso sem olhar a idade pode esconder erros.

## 5. Formatação, filtros e novas variáveis

### 5.1 Padronizando os dados

Vamos fazer quatro ajustes:

- `SEXO`: trocar `F`/`M` por `Feminino`/`Masculino`, para ficar legível;
- `PESO_KG`: arredondar para 1 casa decimal;
- `DT_ATENDIMENTO`: converter de texto para **data de verdade**. Com texto não dá para fazer contas de data ou filtrar por período;
- `NOME` e `UBS`: remover espaços extras no começo e no fim com `.str.strip()`.

In [ ]:
dados['SEXO'] = dados['SEXO'].replace('F', 'Feminino')
dados['SEXO'] = dados['SEXO'].replace('M', 'Masculino')

dados['PESO_KG'] = dados['PESO_KG'].round(1)
dados['DT_ATENDIMENTO'] = pd.to_datetime(dados['DT_ATENDIMENTO'])

dados['NOME'] = dados['NOME'].str.strip()
dados['UBS'] = dados['UBS'].str.strip()
dados.head()

Para confirmar que a data virou data de verdade, olhem o tipo da coluna (deve aparecer `datetime64`):

In [ ]:
dados.dtypes

Por que o `.strip()` importa? Porque para o Python, `"UBS Sul"` e `" UBS Sul"` (com um espaço na frente) são textos **diferentes**. Sem limpar, o gráfico por UBS mostraria duas barras "Sul". Na nossa base, três linhas tinham espaços sobrando, dois nomes e uma UBS.

> **Para pensar:** lembram da duplicata "escondida"? Agora que os espaços foram removidos, rode a célula abaixo. O mesmo paciente aparece duas vezes, no mesmo dia e na mesma UBS, mas com pesos diferentes (um deles com o peso que a gente corrigiu para a mediana). Antes da limpeza o `.drop_duplicates()` não conseguiu enxergar, porque os nomes não eram exatamente iguais. O que vocês fariam com essas duas linhas?

In [ ]:
dados[dados.duplicated(subset=['NOME', 'DT_ATENDIMENTO'], keep=False)]

### 5.2 Maiores e menores valores

`.nlargest(n, coluna)` e `.nsmallest(n, coluna)` mostram as `n` linhas com os maiores e menores valores de uma coluna.

In [ ]:
dados.nlargest(5, 'PESO_KG')

In [ ]:
dados.nsmallest(5, 'PESO_KG')

### 5.3 Filtros

Para filtrar linhas, usamos uma **condição** dentro de colchetes. Primeiro vamos ver o que uma condição devolve sozinha:

In [ ]:
(dados.IDADE >= 60).head()

Ela devolve `True` ou `False` para cada linha. Quando colocamos isso dentro de `dados[ ... ]`, o pandas mantém só as linhas `True`.

Para combinar duas condições usamos `&` (**e**) ou `|` (**ou**), e cada condição vai **entre parênteses**. Exemplo: idosos (60 anos ou mais) atendidos na UBS Norte.

In [ ]:
idosos_ubs_norte = dados[(dados.IDADE >= 60) & (dados.UBS == 'UBS Norte')]
idosos_ubs_norte

Também dá para filtrar por **parte de um texto** com `.str.contains()`. O `na=False` diz que, se a célula estiver vazia, ela deve ser tratada como "não contém" (sem isso, o filtro dá erro quando há ausentes).

In [ ]:
dados[dados.NOME.str.contains('Silva', na=False)]

### 5.4 Criando novas variáveis

**Coluna condicional:** o valor da nova coluna depende de uma condição.

In [ ]:
dados.loc[dados.PESO_KG < 60, 'FAIXA_PESO'] = 'Baixo'
dados.loc[dados.PESO_KG >= 60, 'FAIXA_PESO'] = 'Normal'
dados[['NOME', 'PESO_KG', 'FAIXA_PESO']].head()

> Observação: o corte de 60 kg é só um exemplo para treinar a técnica. Na prática clínica, a classificação de peso depende de outros fatores, como altura e idade (por exemplo, o IMC).

**Coluna calculada:** o valor vem de uma conta com outras colunas. Aqui, peso em gramas.

In [ ]:
dados = dados.assign(PESO_G = dados.PESO_KG * 1000)
dados[['PESO_KG', 'PESO_G']].head()

**Faixas com `pd.cut()`:** transforma uma variável contínua (idade) em categorias. Em `bins` ficam os limites das faixas e em `labels` os nomes. Com `bins=[0, 17, 59, 120]`, as faixas são: acima de 0 até 17, acima de 17 até 59, e acima de 59 até 120.

In [ ]:
dados['FAIXA_ETARIA'] = pd.cut(
    dados.IDADE,
    bins=[0, 17, 59, 120],
    labels=['0-17', '18-59', '60+']
)
dados.head()

In [ ]:
dados['FAIXA_ETARIA'].value_counts()

## 6. Gráficos

Os gráficos servem como uma segunda confirmação do que os números já mostraram. Eles deixam mais evidentes os padrões, as concentrações e os valores extremos.

### 6.1 Histograma da idade

O histograma mostra **como os valores de uma variável numérica se distribuem**. O eixo horizontal é dividido em faixas (as barras) e a altura de cada barra é quantos pacientes caem naquela faixa.

In [ ]:
dados['IDADE'].hist()
plt.title('Distribuicao de idade dos pacientes')
plt.xlabel('Idade')
plt.ylabel('Frequencia')
plt.show()

Olhando o gráfico: não aparece mais nenhuma barra isolada lá longe, em 200 ou em valores negativos. A limpeza funcionou. A maior parte dos atendimentos está entre 20 e 70 anos.

### 6.2 Atendimentos por UBS

`.value_counts()` conta quantas vezes cada categoria aparece e `.plot.bar()` transforma essa contagem em gráfico de barras. É ótimo para comparar categorias.

In [ ]:
dados['UBS'].value_counts().plot.bar()
plt.title('Atendimentos por UBS')
plt.xlabel('UBS')
plt.ylabel('Quantidade de atendimentos')
plt.show()

Agora temos exatamente **três barras**, uma por UBS: Norte (13), Central (12) e Sul (11). Se a gente não tivesse removido os espaços, a UBS Sul apareceria dividida em duas barras.

## 7. Verificação final e exportação

Antes de salvar, repetimos o raio-X do começo: `.info()` e `.describe()`. Conferimos se cada coluna tem o tipo adequado, se ainda existe dado não tratado e se sobrou algum outlier.

In [ ]:
dados.info()

In [ ]:
dados.describe()

In [ ]:
print(dados.shape)
dados.isnull().sum()

Resumo do que aconteceu com a base: começamos com **40 linhas e 6 colunas** e terminamos com **36 linhas e 9 colunas** (3 variáveis novas: `FAIXA_PESO`, `PESO_G` e `FAIXA_ETARIA`). O único ausente que sobrou é o `SEXO` do paciente com código 9, que deixamos de propósito.

### Exportando

Salvamos a versão tratada com **outro nome**, para não perder o arquivo original. O `index=False` evita que o pandas grave a numeração das linhas como uma coluna extra.

In [ ]:
dados.to_csv('pacientes_tratada.csv', index=False)
dados.to_excel('pacientes_tratada.xlsx', index=False)

No Colab, os arquivos ficam na nuvem. Para baixar para o seu computador, rode a célula abaixo (ou use a barra lateral de arquivos).

In [ ]:
from google.colab import files
files.download('pacientes_tratada.csv')
files.download('pacientes_tratada.xlsx')

## Resumo: problema e ferramenta

| Problema na base | O que usamos |
|---|---|
| Separador `;` | `pd.read_csv(..., sep=';')` |
| Linhas repetidas | `.duplicated()` e `.drop_duplicates()` |
| Código "ignorado" (9) | `.replace('9', np.nan)` |
| Valores vazios | `.isnull().sum()`, `.fillna()`, `.dropna()` |
| Valores impossíveis | `.describe()` + `.loc[...] = np.nan` |
| Espaços extras no texto | `.str.strip()` |
| Data como texto | `pd.to_datetime()` |
| Idade em categorias | `pd.cut()` |
| Ver a distribuição | `.hist()` e `.value_counts().plot.bar()` |
| Salvar o resultado | `.to_csv()` e `.to_excel()` |

## Sua vez

Use a base tratada (`dados`) para resolver. Tente primeiro sem olhar o gabarito, que está no final do notebook.

1. Quantos atendimentos existem em cada **faixa etária**?
2. Filtre as **mulheres com mais de 40 anos** atendidas na **UBS Sul**.
3. Faça um **gráfico de barras** com a quantidade de pacientes em cada `FAIXA_PESO`, com título e nomes nos eixos.
4. **Desafio:** encontre a linha do paciente duplicado (aquele da seção 5.1) que ficou com o peso corrigido pela mediana e remova-a da base. Dica: `dados.drop(index=...)` remove uma linha pelo número do índice, que aparece na coluna da esquerda.

In [ ]:
# Exercício 1

In [ ]:
# Exercício 2

In [ ]:
# Exercício 3

In [ ]:
# Exercício 4

---
### Gabarito

Só role daqui para baixo depois de tentar.

In [ ]:
# Gabarito 1
dados['FAIXA_ETARIA'].value_counts()

In [ ]:
# Gabarito 2
dados[(dados.SEXO == 'Feminino') & (dados.IDADE > 40) & (dados.UBS == 'UBS Sul')]

In [ ]:
# Gabarito 3
dados['FAIXA_PESO'].value_counts().plot.bar()
plt.title('Pacientes por faixa de peso')
plt.xlabel('Faixa de peso')
plt.ylabel('Quantidade de pacientes')
plt.show()

In [ ]:
# Gabarito 4
# A linha duplicada com peso preenchido pela mediana tem indice 11 (Marcos Vinicius, peso 69.2).
# A outra (indice 21) tem o peso real, 84.1.
dados = dados.drop(index=11)
dados[dados.NOME == 'Marcos Vinicius']